## EDA & Dataförståelse
Visa datasetstorlek, datatyper och target-fördelning.
Kontrollera saknade värden och beskriv hur ni hanterar dem.
Minst 2 figurer/tabeller + kort tolkning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

## Train/test + preprocessing
Skapa en train/test-split från historical_data.csv.
Bygg en pipeline där preprocessing sker på ett sätt som undviker att testdata påverkar träningen (undvik leakage).
För klassificering: använd gärna stratified split så klasserna fördelas rimligt.

## Modellering och jämförelse
Skapa en baseline.
Träna minst två ytterligare modeller (minst 3 totalt inkl baseline).
Utvärdera med rimlig metod (t.ex. cross-validation på train eller tydligt valideringsupplägg)
Välj metric och motivera valet utifrån ert kravkort.
(Exempel på modeller: LogisticRegression, DecisionTree, RandomForest, GradientBoosting…)

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")

logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

rf = RandomForestClassifier(random_state=42, n_estimators=200, max_features="sqrt", class_weight="balanced")

tree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, class_weight="balanced", random_state=42)


pipe_baseline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", baseline)
])

pipe_logreg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", logreg)
])

pipe_rf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

pipe_tree = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", tree)
])


models = {
    "Baseline": pipe_baseline, 
    "LogisticRegression": pipe_logreg,
    "RandomForest": pipe_rf,
    "DecisionTree": pipe_tree
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "f1"

results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING
    )
    results.append({
        "model": name,
        "mean_f1": scores.mean()
        "std": scores.std()
    })

results_df = pd.DataFrame(results).sort_values("mean_f1", ascending=False)
results_df

best_model_name = results_df.iloc[0]["model"]

best_model = models[best_model_name]

best_model.fit(X_train, y_train)

### Motivering till val av metric

Vi har två kostnader att jobba med: granskningstid och missade bedrägerier. 

* Hög Precision sparar granskningstid, men innebär fler missade bedrägerier
* Hög Recall minskar missade bedrägerier, men innebär högre granskningstid

F1-score är det harmoniska medelvärdet mellan Precision och Recall och ger oss möjligheten att ta hänsyn till båda dessa kostnader då F1-score blir låg om antingen Precision eller Recall är låg. 

## Välj och optimera EN modell (hyperparameter-tuning)

Välj en “final” modell baserat på jämförelsen.
Gör tuning på den valda modellen (litet grid, minst 1–2 parametrar).
Förklara kort vad ni optimerade och varför (koppla till kravkortet).

## Threshold / prioritering (kopplat till kravkortet)

Ni måste bestämma hur modellen ska användas i praktiken. Välj en strategi:

A) Threshold-beslut
flagga misstänkt om proba ≥ t
motivera t utifrån kravkortet och visa konsekvenser (FP/FN eller precision/recall)
B) Top-X prioritering
flagga de X% högst risk (t.ex. top 5% eller top 50 per dag)
motivera X utifrån kravkortet och visa konsekvenser
Ni ska tydligt visa hur ert val påverkar FP/FN och varför det passar er stakeholder.

## Deploy-test: ny data (tisdag kursvecka 6)

När ni får new_data.csv ska ni:
använda er låsta pipeline
skapa prediktioner och en prioriteringslista